# Module 2: Classification using tree-based algorithms

<i>Instructor: Gustavo Sganzerla Martinez</i>


# Lab


In [ ]:
# Code Block
###modules imports

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, roc_auc_score
from scipy.stats import shapiro
import matplotlib.pyplot as plt

In [ ]:
###lets load the cleveland heart disease data set
df = pd.read_csv(
    "data/processed.cleveland.data",
    header = None)

print(df)

###we have 303 rows and 14 columns
###let's name our columns
columns = [
    "age",
    "sex",
    "chest_pain_type",
    "resting_blood_pressure",
    "cholesterol",
    "fasting_blood_sugar",
    "resting_ecg",
    "max_heart_rate",
    "exercise_angina",
    "st_depression",
    "st_slope",
    "num_major_vessels",
    "thalassemia",
    "heart_disease"
]
df.columns = columns

###lets look at the distribution of our independent variable (last column, y)
df['heart_disease'].value_counts()

###in total, there are four classes (0 is no heart disease, 1-4 are different levels)
###we are attempting a binary classification schema, so let's binarize our target

df['binary_outcome'] = (df['heart_disease'] > 0).astype(int)

df['binary_outcome'].value_counts()



In [ ]:
####we need to look for missing values
###in this particular dataset, they encoded "?" as missing value
###lets look in each column for the presence of ?
print((df == "?").sum())

###now we can replace "?" for a NA
df = df.replace("?", np.nan)

print((df == "?").sum())

In [ ]:
###lets create our dependent and independent variables
X = df.drop(columns=['heart_disease', 'binary_outcome'])
y = df['binary_outcome']

print(X.shape)
print(y.shape)

In [ ]:
###now, we need to address the missing values
X.isna().sum()

###let's look for the data types we have in our DF
###note that the columns that originally had the ? are read as object (ie string) and not numeric
###first, we need to address this
df.dtypes

df = df.apply(pd.to_numeric)

df.dtypes

###we have two options: as it is not much data, we can drop the rows
###or we can do an imputation*


In [ ]:
###before imputting, lets look at the distribution of these two columns

temp_num_major_vessels = df['num_major_vessels'].dropna()

shapiro_num_major_vessels = shapiro(temp_num_major_vessels)
print(shapiro_num_major_vessels.pvalue)

####low p-value, non-normal

temp_thalassemia = df['thalassemia'].dropna()

shapiro_thalassemias = shapiro(temp_thalassemia)
print(shapiro_thalassemias.pvalue)

####low p-value, non-normal

In [ ]:
###now, we need to create a train and a test split
###the model first looks for patterns in the train data
###we evaluate the model on the test data


X = df.drop(columns=['heart_disease', 'binary_outcome'])
y = df['binary_outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    random_state=42,
    stratify = y
)

###the arguments we are passing to the function
###X and y datasets
###test size
###random state
###stratify, to preserve the class proportions in train and test

###let's check
print("Full dataset:")
print(y.value_counts(normalize=True))

print("Train dataset:")
print(y_train.value_counts(normalize=True))

print("Test dataset:")
print(y_test.value_counts(normalize=True))


In [ ]:
###both columns are not normally distributted, we better represent that with median

###instantiate the imputer
imputer = SimpleImputer(strategy="median")

###now a very important step, in imputting, we are learning the median of the dataset and adding it to the missing
###if we do in the full dataset, we might introduce data leakage, where the test now has information from the train
###so lets obtain the mdian on the train only and imput it in the train/test

X_train_imputted = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns = X_train.columns,
    index=X_train.index
)

X_test_imputted = pd.DataFrame(
    imputer.transform(X_test),
    columns = X_test.columns,
    index=X_test.index
)

In [ ]:
###now we are ready to create our decision tree


###first we instantiate our tree
dt = DecisionTreeClassifier(
    random_state=42
)

###then we fit the training data
dt.fit(X_train_imputted, y_train)

###first, lets look at the tree itself

print("Tree depth:", dt.get_depth())
print("Number of leaves:", dt.get_n_leaves())
print(dt.get_params())


###when we instantiated our tree, we pretty much used the default option
##some of the parameters that might be worth checking

##criterion = how the quality of a split is evaluated
###max_depth = how much we allow the tree to grow
###min_samples_split = a node only need 2 samples for splitting
###min_samples_leaf = a terminal leaf can contain only 1 observation

###unrescricted trees can overfit quite aggressively as we allow single leaf nodes

train_accuracy = dt.score(X_train_imputted, y_train)
test_accuracy = dt.score(X_test_imputted, y_test)

print("Training accuracy:", train_accuracy)
print("Testing accuracy:", test_accuracy)




In [ ]:
###we can iterate over multiple depths

for depth in [1, 2, 3, 4, 5, 10]:

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train_imputted, y_train)

    train_score = model.score(
        X_train_imputted,
        y_train
    )

    test_score = model.score(
        X_test_imputted,
        y_test
    )

    print(
        depth,
        round(train_score, 3),
        round(test_score, 3)
    )

###it looks like depth 3 has a good balance

In [ ]:
###lets plot our tree


dt_depth3 = DecisionTreeClassifier(
    max_depth = 3,
    random_state=42
)
dt_depth3.fit(X_train_imputted, y_train)


plt.figure(figsize=(20, 10))

plot_tree(
    dt_depth3,
    feature_names=X.columns,
    class_names=[str(c) for c in sorted(y.unique())],
    filled=True,
    rounded=True,
    fontsize=10
)

plt.show()


In [ ]:
y_pred = dt_depth3.predict(X_test_imputted)


ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    cmap="Blues"
)

plt.title("Decision Tree — Confusion Matrix")
plt.show()

In [ ]:
###now into random forest

###a random forest is a collection of decision trees

###there are five important parameters for us to consider:
##n_estimators = how many trees (+ generally more stable but plateaus at somepoint. no impact in overfitting. + trees = + computational cost)

##max_depth = how complex can each tree become

##max_features = how different are the trees?
###not every tree is trained with the same features, if they all had access to the same features, a few strong predictors would cause many trees to look similar
###it decorrelates the trees, at every splot, a random subset of features is considered

##min_samples_leaf - how small can terminal groups become (we avoid having a leaf node with a single/too few observations)

##class_weight - what if classes are imbalanced?


###lets find the best combination of these parameters


###first, lets instantiate an empty RFClassifier object

rf = RandomForestClassifier(
    random_state=42)

###now we set the grid of parameters

param_grid = {
    "n_estimators": [10, 25, 50, 100],
    "max_depth": [5, 10, 20],
    "max_features": ["sqrt", "log2"],
    "min_samples_leaf": [1, 2, 5]
}

grid_search = GridSearchCV(
    rf, ##the model
    param_grid, ##the grid we defined
    scoring="accuracy",
    n_jobs=-1, ###for paralellization, -1 uses all available processors,
    verbose=1
)

##now we fit our training data into it

grid_search.fit(X_train, y_train)

print("Best parameters: ", grid_search.best_params_)
print("Cross-validation accuracy: ", grid_search.best_score_)

###now instead of instantiating manually an RFC with the best params, we can automatically obtain them

best_rf = grid_search.best_estimator_

###lets make predictions using our estimator
y_pred = best_rf.predict(X_test)


In [ ]:
###let's evaluate our estimator

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn/(tn+fp)

print(f"Accuracy:    {accuracy:.3f}")
print(f"Precision:   {precision:.3f}")
print(f"Sensitivity: {sensitivity:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"F1-score:    {f1:.3f}")



# Predicted probability for class 1
y_prob = best_rf.predict_proba(X_test)[:, 1]

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# AUC
auc = roc_auc_score(y_test, y_prob)

print(f"ROC-AUC: {auc:.3f}")

plt.figure(figsize=(7, 6))

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"Random Forest (AUC = {auc:.3f})"
)

# Random classifier reference
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)



plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC Curve – Random Forest")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
###RF feature importance

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_rf.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance_plot = importance.sort_values(
    "Importance",
    ascending=True
)

plt.figure(figsize=(8, 6))

plt.barh(
    importance_plot["Feature"],
    importance_plot["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")

plt.tight_layout()
plt.show()